<a href="https://colab.research.google.com/github/jamie07262/INFO-3608-PROJECT/blob/main/XGBoostWithNewDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
import requests
import re
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss
import matplotlib.pyplot as plt

In [69]:
import os
import requests

# GitHub API URL for the directory
api_url = "https://api.github.com/repos/jamie07262/INFO-3608-PROJECT/contents/data/processed"

# Create local directory
os.makedirs("processed", exist_ok=True)

# Get list of files from GitHub API
response = requests.get(api_url)
if response.status_code == 200:
    files = response.json()
    for file in files:
        if file['name'].endswith('.csv'):
            download_url = file['download_url']
            r = requests.get(download_url)
            with open(f"processed/{file['name']}", "wb") as f:
                f.write(r.content)
            print(f"Downloaded: {file['name']}")
else:
    print(f"Failed to access API: {response.status_code}")

print("All CSV files downloaded.")

Downloaded: Features.csv
Downloaded: NewFeaturesForPredictions.csv
Downloaded: Predictions.csv
Downloaded: SampleSubmissionStage2.csv
All CSV files downloaded.


In [70]:
df = pd.read_csv("processed/NewFeaturesForPredictions.csv")
df_features = df.copy()

In [71]:
# FEATURES = [
#     # 'Season', 'T1TeamID', 'T2TeamID',
#         'T1AverageScore', 'T2AverageScore',
#     # 'PointDifference',
#       #       'T1Seed', 'T2Seed', 'SeedDifference',
#       #   'T1Rating', 'T2Rating', 'RatingDifference',
#       #       'T1Grade', 'T2Grade', 'GradeDifference',
#        'Divison',
#             # 'DivisonLabel',
#             # 'Win',
#             # 'WinLabel',
#        'T1AverageFGM',
#        'T1AverageFGA', 'T1AverageFGM3', 'T1AverageFGA3', 'T1AverageFTM',
#        'T1AverageFTA', 'T1AverageOR', 'T1AverageDR', 'T1AverageAst',
#        'T1AverageTO', 'T1AverageStl', 'T1AverageBlk', 'T1AveragePF',
#        'T1AverageOpponentScore', 'T1AverageOpponentFGM',
#        'T1AverageOpponentFGA', 'T1AverageOpponentFGM3',
#        'T1AverageOpponentFGA3', 'T1AverageOpponentFTM', 'T1AverageOpponentFTA',
#        'T1AverageOpponentOR', 'T1AverageOpponentDR', 'T1AverageOpponentAst',
#        'T1AverageOpponentTO', 'T1AverageOpponentStl', 'T1AverageOpponentBlk',
#        'T1AverageOpponentPF', 'T1AveragePointDifference',
#         'T2AverageFGM',
#        'T2AverageFGA', 'T2AverageFGM3', 'T2AverageFGA3', 'T2AverageFTM',
#        'T2AverageFTA', 'T2AverageOR', 'T2AverageDR', 'T2AverageAst',
#        'T2AverageTO', 'T2AverageStl', 'T2AverageBlk', 'T2AveragePF',
#        'T2AverageOpponentScore', 'T2AverageOpponentFGM',
#        'T2AverageOpponentFGA', 'T2AverageOpponentFGM3',
#        'T2AverageOpponentFGA3', 'T2AverageOpponentFTM', 'T2AverageOpponentFTA',
#        'T2AverageOpponentOR', 'T2AverageOpponentDR', 'T2AverageOpponentAst',
#        'T2AverageOpponentTO', 'T2AverageOpponentStl', 'T2AverageOpponentBlk',
#        'T2AverageOpponentPF', 'T2AveragePointDifference']
FEATURES = ["Divison", "T1Seed", "T2Seed", "SeedDifference",
            "T1AverageScore", "T1AverageFGA", "T1AverageOR", "T1AverageDR",
            "T1AverageBlk", "T1AveragePF", "T1AverageOpponentFGA",
            "T1AverageOpponentBlk", "T1AverageOpponentPF",
            "T1AveragePointDifference",
            "T2AverageScore", "T2AverageFGA", "T2AverageOR", "T2AverageDR",
            "T2AverageBlk", "T2AveragePF", "T2AverageOpponentFGA", "T2AverageOpponentBlk", "T2AverageOpponentPF", "T2AveragePointDifference",
            "T1Rating", "T2Rating", "RatingDifference", "T1Grade", "T2Grade" ]

In [72]:
import xgboost as xgb

param = {}
param["objective"] = "reg:squarederror"
param["booster"] = "gbtree"
param["eta"] = 0.0093  #0.009 #0.01
param["subsample"] = 0.6
param["colsample_bynode"] = 0.8
param["num_parallel_tree"] = 2
param["min_child_weight"] = 4
param["max_depth"] = 4
param["tree_method"] = "hist"
param['grow_policy'] = 'lossguide'
param["max_bin"] = 38   #32

num_rounds = 704 #704 #700

In [73]:
from sklearn.metrics import mean_absolute_error, brier_score_loss

models = {}
oof_mae = []
oof_preds = []
oof_targets = []
oof_ss = []

# leave-one-season out models
for season in set(df_features["Season"]):
    x_train = df_features.loc[df_features["Season"] != season, FEATURES].values
    y_train = df_features.loc[df_features["Season"] != season, "PointDifference"].values
    x_val = df_features.loc[df_features["Season"] == season, FEATURES].values
    y_val = df_features.loc[df_features["Season"] == season, "PointDifference"].values
    s_val = df_features.loc[df_features["Season"] == season, "Season"].values

    dtrain = xgb.DMatrix(x_train, label=y_train)
    dval = xgb.DMatrix(x_val, label=y_val)
    models[season] = xgb.train(
        params=param,
        dtrain=dtrain,
        num_boost_round = num_rounds,
    )
    preds = models[season].predict(dval)
    print(f"oof season {season} mae: {mean_absolute_error(y_val, preds)}")
    oof_mae.append(mean_absolute_error(y_val, preds))
    oof_preds += list(preds)
    oof_targets += list(y_val)
    oof_ss += list(s_val)

print(f"average mae: {np.mean(oof_mae)}")

oof season 2003 mae: 8.855619733403243
oof season 2004 mae: 7.890549957977944
oof season 2005 mae: 7.794355535287307
oof season 2006 mae: 8.698771061164896
oof season 2007 mae: 7.633056669009642
oof season 2008 mae: 9.809866485622479
oof season 2009 mae: 9.135754582300109
oof season 2010 mae: 8.675639155865603
oof season 2011 mae: 9.681133770899386
oof season 2012 mae: 8.407861776723184
oof season 2013 mae: 9.992138884586847
oof season 2014 mae: 10.169756133650475
oof season 2015 mae: 7.778544812553969
oof season 2016 mae: 10.211964269405605
oof season 2017 mae: 9.883130741975885
oof season 2018 mae: 10.303126421882812
oof season 2019 mae: 8.910894716129853
oof season 2021 mae: 10.596119731910122
oof season 2022 mae: 10.41047442453206
oof season 2023 mae: 9.479419143156537
oof season 2024 mae: 9.357544003999722
average mae: 9.22265342914465


In [74]:
from itertools import product


teams = pd.unique(df[["T1TeamID", "T2TeamID"]].values.ravel())


matchups = set()
year = 2025

for t1, t2 in product(teams, repeat=2):
    if t1 != t2:
        low, high = sorted([t1, t2])
        matchups.add(f"{year}_{low}_{high}")


print(len(matchups))
matchups = list(matchups)
matchups = sorted(matchups)
print(matchups[121277])

121278
2025_3462_3465


In [75]:
print((df[df['T1TeamID'] == 3479] | df['T2TeamID'] == 3479))

Empty DataFrame
Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, ...]
Index: []

[0 rows x 4627 columns]
